# 01 · EDA and RFM
E-Commerce Customer Segmentation — ADS Mini Project (Group 17)

Run this notebook from the `notebooks/` folder. It uses the real dataset if `data/raw/Online Retail.xlsx` exists, otherwise the sample.

In [ ]:
import sys; sys.path.append('..')
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from src.cleaning import load_raw, clean
from src.rfm import build_rfm
from src.clustering import *
sns.set_theme()

raw_path = Path('../data/raw/Online Retail.xlsx')
if not raw_path.exists():
    raw_path = Path('../data/sample/sample_online_retail.csv')
raw = load_raw(raw_path)
raw.head()

## 1. Data understanding

In [ ]:
print(raw.shape)
raw.info()
raw.isna().sum()

In [ ]:
raw.describe()

## 2. Cleaning
Drop rows with no CustomerID, cancelled invoices (start with `C`), non-positive quantity/price and duplicates.

In [ ]:
df, report = clean(raw)
pd.Series(report)

## 3. EDA

In [ ]:
monthly = df.set_index('InvoiceDate')['TotalPrice'].resample('MS').sum()
monthly.plot(marker='o', title='Monthly revenue', figsize=(9,4)); plt.show()

In [ ]:
top = df.groupby('Country')['TotalPrice'].sum().nlargest(10)
top.plot.barh(title='Top countries by revenue', figsize=(8,4)).invert_yaxis(); plt.show()

## 4. RFM features

In [ ]:
rfm = build_rfm(df)
rfm.describe()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
for a, c in zip(ax, ['Recency','Frequency','Monetary']):
    sns.histplot(rfm[c], bins=40, ax=a); a.set_title(c)
plt.tight_layout(); plt.show()

RFM values are right-skewed, so we apply `log1p` and standardise before K-Means.

In [ ]:
X = prepare_features(rfm)
metrics = evaluate_k(X)
metrics

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].plot(metrics.k, metrics.inertia, marker='o'); ax[0].set_title('Elbow')
ax[1].plot(metrics.k, metrics.silhouette, marker='o', color='g'); ax[1].set_title('Silhouette')
plt.show()

## 5. Final K-Means model

In [ ]:
K = 4
seg = add_pca(X, name_segments(rfm, fit_kmeans(X, K)))
segment_summary(seg)

In [ ]:
sns.scatterplot(data=seg, x='PC1', y='PC2', hue='Segment', s=18, palette='Set2')
plt.title('Customer segments (PCA view)'); plt.show()

## 6. Business interpretation
Write your findings here: which segment earns the most revenue, how large it is, and what action you recommend for each segment.